# CHAPTER 10 - Data Aggregation and Group Operations

----

In [50]:
%run "Ch00 - Basic Imports and Set ups.ipynb"

Categorizing a dataset and applying a function to each group, whether an aggregation
or transformation, is often a critical component of a data analysis workflow. After
loading, merging, and preparing a dataset, you may need to compute group statistics
or possibly pivot tables for reporting or visualization purposes. pandas provides a
flexible `groupby` interface, enabling you to slice, dice, and summarize datasets in a
natural way.

One reason for the popularity of relational databases and SQL (which stands for
“structured query language”) is the ease with which data can be joined, filtered, transformed,
and aggregated. However, query languages like SQL are somewhat constrained
in the kinds of group operations that can be performed. As you will see, with
the expressiveness of Python and pandas, we can perform quite complex group operations
by utilizing any function that accepts a pandas object or NumPy array. In this
chapter, you will learn how to:

    • Split a pandas object into pieces using one or more keys (in the form of functions, arrays, or DataFrame column names)
    • Calculate group summary statistics, like count, mean, or standard deviation, or a user-defined function    
    • Apply within-group transformations or other manipulations, like normalization, linear regression, rank, or subset selection
    • Compute pivot tables and cross-tabulations
    • Perform quantile analysis and other statistical group analyses

----

Aggregation of time series data, a special use case of groupby, is
referred to as resampling in this book and will receive separate
treatment in Chapter 11.

## Basic Imports and Set ups

In [6]:
%matplotlib inline
import pandas as pd
import numpy as np
import os
from faker import Faker
import matplotlib.pyplot as plt
from datetime import datetime
from IPython.display import display, HTML
import seaborn as sns

In [7]:
import pandas as pd
from IPython.display import display, HTML

def render_book_table(title, columns, rows):
    """
    Render a documentation-style table in Jupyter.

    Parameters
    ----------
    title : str
        Title displayed above the table
    columns : list
        Column names
    rows : list of lists
        Table data rows
    """

    table_data = pd.DataFrame(rows, columns=columns)

    styled_table = (
        table_data.style
        .hide(axis="index")
        .set_table_styles([
            {"selector": "th",
             "props": [("background-color", "#f2f2f2"),
                       ("font-weight", "bold"),
                       ("text-align", "center"),
                       ("border", "1px solid #999"),
                       ("padding", "8px")]},

            {"selector": "td",
             "props": [("border", "1px solid #999"),
                       ("padding", "8px"),
                       ("white-space", "normal"),
                       ("text-align", "left")]},

            {"selector": "table",
             "props": [("border-collapse", "collapse"),
                       ("width", "100%")]}
        ])
    )

    display(HTML(f"<h3>{title}</h3>"))
    display(styled_table)



## 10.1 GroupBy Mechanics

Hadley Wickham, an author of many popular packages for the R programming language,
coined the term split-apply-combine for describing group operations. In the
first stage of the process, data contained in a pandas object, whether a Series, Data‐
Frame, or otherwise, is split into groups based on one or more keys that you provide.
The splitting is performed on a particular axis of an object. For example, a DataFrame
can be grouped on its rows (`axis=0`) or its columns (`axis=1`). Once this is done, a
function is applied to each group, producing a new value. Finally, the results of all
those function applications are combined into a result object. The form of the resulting
object will usually depend on what’s being done to the data. See Figure BELOW for a
mockup of a simple group aggregation.

<img src="images/ch10-1-group.jpg" width="500" alt="Figure 10-1. Illustration of a group aggregation" />

Each grouping key can take many forms, and the keys do not have to be all of the
same type:

    • A list or array of values that is the same length as the axis being grouped
    • A value indicating a column name in a DataFrame
    • A dict or Series giving a correspondence between the values on the axis being grouped and the group names
    • A function to be invoked on the axis index or the individual labels in the index

Note that the latter three methods are shortcuts for producing an array of values to be
used to split up the object. Don’t worry if this all seems abstract. Throughout this
chapter, I will give many examples of all these methods. To get started, here is a small
tabular dataset as a DataFrame:

In [8]:
df = pd.DataFrame({'key1' : ['a', 'a', 'b', 'b', 'a'],
                   'key2' : ['one', 'two', 'one', 'two', 'one'],
                   'data1' : np.random.randn(5),
                   'data2' : np.random.randn(5)})

In [9]:
df

,key1,key2,data1,data2
0,a,one,2.084528,2.305648
1,a,two,0.819379,0.812119
2,b,one,-0.106112,0.208885
3,b,two,0.154015,1.458932
4,a,one,0.366758,-0.462591


Suppose you wanted to compute the mean of the data1 column using the labels from
`key1`. There are a number of ways to do this. One is to access data1 and call groupby
with the column (a Series) at `key1`:

In [10]:
grouped = df['data1'].groupby(df['key1'])

In [11]:
grouped

This grouped variable is now a GroupBy object. It has not actually computed anything
yet except for some intermediate data about the group key `df['key1']`. The idea is
that this object has all of the information needed to then apply some operation to
each of the groups. For example, to compute group means we can call the GroupBy’s
`mean` method:

In [12]:
grouped.mean()

key1
a    1.090222
b    0.023952
Name: data1, dtype: float64

Later, I’ll explain more about what happens when you call `.mean()`. The important
thing here is that the data (a Series) has been aggregated according to the group key,
producing a new Series that is now indexed by the unique values in the `key1` column.

The result index has the name `'key1'` because the DataFrame column `df['key1']`
did.

If instead we had passed multiple arrays as a list, we’d get something different:

In [13]:
means = df['data1'].groupby([df['key1'], df['key2']]).mean()

In [14]:
means

key1  key2
a     one     1.225643
      two     0.819379
b     one    -0.106112
      two     0.154015
Name: data1, dtype: float64

Here we grouped the data using two keys, and the resulting Series now has a hierarchical
index consisting of the unique pairs of keys observed:

In [16]:
means.unstack()

key2,one,two
key1,,
a,1.225643,0.819379
b,-0.106112,0.154015


In this example, the group keys are all Series, though they could be any arrays of the
right length:

In [17]:
states = np.array(['Ohio', 'California', 'California', 'Ohio', 'Ohio'])

In [18]:
years = np.array([2005, 2005, 2006, 2005, 2006])

In [19]:
df['data1'].groupby([states, years]).mean()

California  2005    0.819379
            2006   -0.106112
Ohio        2005    1.119271
            2006    0.366758
Name: data1, dtype: float64

Note above , the hierarchical index is build on the group by data set values, not on any column in data frame. 

Frequently the grouping information is found in the same DataFrame as the data you
want to work on. In that case, you can pass column names (whether those are strings,
numbers, or other Python objects) as the group keys:  ( you dont have to use full `dataframe[<column name>]` format)



In [21]:
df

,key1,key2,data1,data2
0,a,one,2.084528,2.305648
1,a,two,0.819379,0.812119
2,b,one,-0.106112,0.208885
3,b,two,0.154015,1.458932
4,a,one,0.366758,-0.462591


In [22]:
df.groupby('key1').mean()

TypeError: agg function failed [how->mean,dtype->object]

In [23]:
df.groupby('key1').mean(numeric_only=True)

,data1,data2
key1,,
a,1.090222,0.885059
b,0.023952,0.833908


##### ✅ Root Cause of This Pandas Error (Very Important Concept)

This error is **NOT a bug in pandas** 🙂
It is a **data-type problem during aggregation.**

---

##### ⭐ What the Error Really Says

The key line is:

```
TypeError: Could not convert string 'onetwoone' to numeric
```

And later:

```
agg function failed [how->mean, dtype->object]
```

This means:

👉 You are doing:

```python
df.groupby('key1').mean()
```

But **at least one column inside `df` is of type `object` (string)**
and pandas is trying to compute **mean on that column.**

---

##### ⭐ Why Pandas Is Trying To Do Mean On Strings

In modern pandas versions:

When you run:

```python
df.groupby('key1').mean()
```

pandas tries to aggregate **ALL columns** unless told otherwise.

If a column contains values like:

```
one
two
one
```

Then internally pandas tries something like:

```
"one" + "two" + "one"
→ "onetwoone"
```

Then it tries to convert that into numeric → 💥 ERROR.

---

##### ⭐ Example That Produces Same Error

```python
df = pd.DataFrame({
    'key1': ['a','a','b'],
    'val1': [1,2,3],
    'val2': ['one','two','one']
})

df.groupby('key1').mean()
```

❌ Boom → same error.

---

##### ✅ Correct Ways To Fix It (Senior DS Level Understanding)

##### ⭐ Fix 1 — Select Only Numeric Columns (Best Practice)

This is the **cleanest real-world solution.**

```python
df.groupby('key1')[['val1']].mean()
```

OR

```python
df.groupby('key1').mean(numeric_only=True)
```

---

##### ⭐ Fix 2 — Convert Column To Numeric (If It Should Be Numeric)

If column is wrongly typed:

```python
df['val2'] = pd.to_numeric(df['val2'], errors='coerce')
```

---

##### ⭐ Fix 3 — Use Different Aggregation For Strings

Sometimes in DS we WANT string aggregation.

Example:

```python
df.groupby('key1').agg({
    'val1': 'mean',
    'val2': 'first'
})
```

---

##### ⭐ VERY Important Data Science Insight (Interview Level)

##### 🔥 GroupBy Aggregations Require Semantic Meaning

You must always ask:

> “Does mean make sense for this column?”

| Column Type     | Valid Aggregation  |
| --------------- | ------------------ |
| numeric         | mean, sum, std     |
| category/string | count, first, mode |
| datetime        | min, max           |

This is a **core Data Science maturity signal.**

---



You may have noticed in the first case `df.groupby('key1').mean()` that there is no
`key2` column in the result. Because `df['key2']` is not numeric data, it is said to be a
nuisance column, which is therefore excluded from the result. By default, all of the
numeric columns are aggregated, though it is possible to filter down to a subset, as
you’ll see soon.
Regardless of the objective in using groupby, a generally useful GroupBy method is
size, which returns a Series containing group sizes:

# 🎯
> Note that, at the time of this notebook (Mar 2026), the nuisance columns can be removed only if you use `numeric_only=True`

Regardless of the objective in using groupby, a generally useful GroupBy method is
`size`, which returns a Series containing group sizes:

In [25]:
df.groupby(['key1', 'key2']).size()

key1  key2
a     one     2
      two     1
b     one     1
      two     1
dtype: int64

In [27]:
df.groupby('key1').size()

key1
a    3
b    2
dtype: int64

Take note that any missing values in a group key will be excluded from the result.

---

### Iterating Over Groups

The GroupBy object supports iteration, generating a sequence of 2-tuples containing
the group name along with the chunk of data. Consider the following:

In [28]:
for name, group in df.groupby('key1'):
    print(name)
    print(group)

a
  key1 key2     data1     data2
0    a  one  2.084528  2.305648
1    a  two  0.819379  0.812119
4    a  one  0.366758 -0.462591
b
  key1 key2     data1     data2
2    b  one -0.106112  0.208885
3    b  two  0.154015  1.458932


In the case of multiple keys, the first element in the tuple will be a tuple of key values:

In [35]:
for (name1,name2), group in df.groupby(['key1', 'key2']) :
    print(name1,name2)
    print(group)

a one
  key1 key2     data1     data2
0    a  one  2.084528  2.305648
4    a  one  0.366758 -0.462591
a two
  key1 key2     data1     data2
1    a  two  0.819379  0.812119
b one
  key1 key2     data1     data2
2    b  one -0.106112  0.208885
b two
  key1 key2     data1     data2
3    b  two  0.154015  1.458932


Of course, you can choose to do whatever you want with the pieces of data. A recipe
you may find useful is computing a dict of the data pieces as a one-liner:

In [36]:
pieces = dict(list(df.groupby('key1')))

In [37]:
pieces

{'a':   key1 key2     data1     data2
 0    a  one  2.084528  2.305648
 1    a  two  0.819379  0.812119
 4    a  one  0.366758 -0.462591,
 'b':   key1 key2     data1     data2
 2    b  one -0.106112  0.208885
 3    b  two  0.154015  1.458932}

In [38]:
pieces['b']

,key1,key2,data1,data2
2,b,one,-0.106112,0.208885
3,b,two,0.154015,1.458932


(a dict of collection of data frames group ed by columns of your interest)

By default `groupby` groups on `axis=0`, but you can group on any of the other axes.
For example, we could group the columns of our example `df` here by `dtype` like so:

In [39]:
df.dtypes

key1      object
key2      object
data1    float64
data2    float64
dtype: object

In [40]:
df

,key1,key2,data1,data2
0,a,one,2.084528,2.305648
1,a,two,0.819379,0.812119
2,b,one,-0.106112,0.208885
3,b,two,0.154015,1.458932
4,a,one,0.366758,-0.462591


In [41]:
grouped = df.groupby(df.dtypes, axis=1)

C:\Users\212364780.HCAD\AppData\Local\Temp\ipykernel_11144\521057107.py:1: FutureWarning: DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.
  grouped = df.groupby(df.dtypes, axis=1)


In [42]:
for dtype, group in grouped:
    print(dtype)
    print(group)

float64
      data1     data2
0  2.084528  2.305648
1  0.819379  0.812119
2 -0.106112  0.208885
3  0.154015  1.458932
4  0.366758 -0.462591
object
  key1 key2
0    a  one
1    a  two
2    b  one
3    b  two
4    a  one


(this can be used to segragate various kind of data types in a frame).

Another way to do the same ( Above one gave deprecation warning).

In [47]:
grouped = df.T.groupby(df.dtypes)

for dtype, group in grouped:
    print(dtype)
    print(group.T)

float64
      data1     data2
0  2.084528  2.305648
1  0.819379  0.812119
2 -0.106112  0.208885
3  0.154015  1.458932
4  0.366758 -0.462591
object
  key1 key2
0    a  one
1    a  two
2    b  one
3    b  two
4    a  one


##### ✅ Why You Are Getting This Warning

You wrote:

```python
grouped = df.groupby(df.dtypes, axis=1)
```

This means:

👉 **Group columns (NOT rows)** based on their dtype.

Earlier pandas allowed:

```
groupby(..., axis=1)
```

But now this is **deprecated**.

So pandas shows warning:

```
FutureWarning: DataFrame.groupby with axis=1 is deprecated.
Do frame.T.groupby(...) without axis instead.
```

---

##### ✅ Correct Modern Usage (Very Important)

You must **transpose first → then group normally.**

```python
grouped = df.T.groupby(df.dtypes)
```

---

##### ⭐ Why This Works

Original intention:

> Group columns by dtype

But pandas wants **groupby to always work row-wise (axis=0)**

So we convert:

| Step                 | Meaning                                        |
| -------------------- | ---------------------------------------------- |
| `df.T`               | Columns become rows                            |
| `groupby(df.dtypes)` | Now grouping those “rows” (= original columns) |

---

##### ⭐ Full Example (Your Case)

```python
grouped = df.T.groupby(df.dtypes)

for dtype, group in grouped:
    print(dtype)
    print(group)
```

---

##### ⭐ Even Cleaner Senior DS Style

Usually we also store dtype mapping first:

```python
col_types = df.dtypes

grouped = df.T.groupby(col_types)
```

Very readable in production notebooks.

---

##### ⭐ What This Grouping Actually Produces

Your dataframe has:

| Column | dtype  |
| ------ | ------ |
| key1   | object |
| key2   | object |
| data1  | float  |
| data2  | float  |

So grouping result becomes:

```
object → key1, key2
float64 → data1, data2
```

This is extremely useful for:

✅ numeric preprocessing
✅ categorical encoding
✅ automated pipelines

---

##### 🚀 Senior Architect Insight

Modern pandas direction is:

> ❗ Remove axis parameter from many APIs
> ❗ Force consistent row-wise mental model

So future-safe patterns are:

| Old                    | New                 |
| ---------------------- | ------------------- |
| `groupby(..., axis=1)` | `df.T.groupby(...)` |
| `sum(axis=1)`          | still OK            |
| `apply(axis=1)`        | still OK            |

But **groupby(axis=1) → going away.**

---

